In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd

from nucleitracking.pipeline.config import PipelineConfig

base_path = Path(r"C:\Tracking\NucleiTracking\data\interim\lightsheet\final_processing")

ss = pd.read_excel(base_path / "final_processing.xlsx", sheet_name="Sheet1")

ss

In [ ]:
configs_path = base_path / "CONFIGS"

base_config = PipelineConfig.load(configs_path / "base.yml")
configs = []

for embryo in ss["Embryo"].unique():
    if pd.isna(embryo):
        continue

    if embryo != "20250131_polecells":
        continue

    embryo = str(embryo)
    config_path = configs_path / f"{embryo}.yml"

    configs.append(config_path)

    if config_path.exists():
        print(f"Config for {embryo} already exists, skipping.")
        continue

    embryo_config: PipelineConfig = base_config.model_copy()

    embryo_config.dataset = base_path / embryo
    embryo_config.param_set_name = embryo

    embryo_config.save(config_path)

In [ ]:
from nucleitracking.pipeline.runner import PipelineRunner

for config in configs:
    runner = PipelineRunner(config)
    runner.run(phase="local_pre")

In [ ]:
from nucleitracking.pipeline.runner import PipelineRunner
from nucleitracking.pipeline.steps.local_post import (
    run_merge_3d_tracking,
    run_merge_centroids,
)

for config in configs:
    print(config)
    config = PipelineConfig.load(config)
    run_merge_centroids(config.dataset, config)
    run_merge_3d_tracking(config.dataset, config)

In [ ]:
import napari
from tqdm import tqdm

viewer = napari.Viewer()
for config in tqdm(configs):
    config = PipelineConfig.load(config)
    centroids_path = (
        config.dataset / f"tracking_{config.param_set_name}" / "new_centroids.csv"
    )
    if not centroids_path.exists():
        continue

    centroids = pd.read_csv(centroids_path)
    viewer.add_points(
        data=centroids[["frame", "px_x", "px_y", "px_z"]].values,
        name=config.param_set_name,
    )

napari.run()

In [ ]:
import napari

from nucleitracking.pipeline.runner import PipelineConfig
from nucleitracking.pipeline.steps import local_post

# viewer = napari.Viewer()

for config in configs:
    print(config)
    config = PipelineConfig.load(config)
    # local_post.run_merge_centroids(config.dataset, config)
    local_post.run_tracking(config.dataset, config)
    local_post.run_division_mapping(config.dataset, config)
    local_post.run_set_cylindrical_coords(config.dataset, config, None)
    local_post.run_export_hdf5(config.dataset, config)

# napari.run()

In [ ]:
from shutil import copyfile

export_path = base_path / "exported_spots"
export_path.mkdir(exist_ok=True)

for config in configs:
    print(config)
    config = PipelineConfig.load(config)
    h5_path = (
        config.dataset
        / f"tracking_{config.param_set_name}"
        / f"{config.param_set_name}_spots.h5"
    )
    if not h5_path.exists():
        print(f"HDF5 file for {config.param_set_name} does not exist, skipping.")
        continue

    copyfile(h5_path, export_path / f"{config.param_set_name}_spots.h5")